# Notebook 06: Synthetic Phi Recovery

This notebook validates the X-Theta fitting pipeline by generating synthetic Bell-test data with a known $\Phi$ phase and attempting to recover it.

In [ ]:
from __future__ import annotations
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path
import matplotlib.pyplot as plt

# Standardized project root addition
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from xtheta.data.adapters.synthetic import generate_synthetic_xtheta_data
from xtheta.data.validation import run_open_data_chsh_validation

## 1. Run Recovery for Multiple Phi Values

We test a range of $\Phi$ values from 0.0 to 0.3.

In [ ]:
phi_values = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
all_results = []

for phi in phi_values:
    data_iterator = generate_synthetic_xtheta_data(phi, n_trials=10000, seed=42)
    res = run_open_data_chsh_validation(
        data_iterator,
        dataset_name=f"synthetic_{phi:.2f}",
        output_dir="../outputs/synthetic_validation",
        bootstrap_samples=0, # No bootstrap for speed
        claim_level="simulation"
    )
    res['phi_true'] = phi
    all_results.append(res)

df = pd.DataFrame(all_results)
df[['phi_true', 'phi_eff', 'CHSH_S', 'R_theta_eff']]

## 2. Visualize Recovery Accuracy

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(df['phi_true'], df['phi_eff'], 'o-', label='Recovered')
plt.plot([0, 0.3], [0, 0.3], 'k--', label='Ideal')
plt.xlabel('True Phi')
plt.ylabel('Recovered Phi_eff')
plt.title('Synthetic Phi Recovery Validation')
plt.legend()
plt.grid(True)
plt.show()

## 3. CHSH S Variation

Show how the CHSH S-statistic decreases as $\Phi$ increases.

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(df['phi_true'], df['CHSH_S'], 's-')
plt.xlabel('True Phi')
plt.ylabel('CHSH S')
plt.title('CHSH S vs Relational Phase Phi')
plt.grid(True)
plt.show()